# Data QC — GSE218989 & GSE207422

Verifies that the loaded datasets match the statistics reported in:
> Mubarak et al. *Unpacking Genomic Biomarkers for PD-1 Immunotherapy Success
> in NSCLC Using Deep Neural Networks.* JMIR Bioinform Biotech 2026;7:e70553.

**Expected (from paper)**

| Cohort | Samples | Genes | Responders | Non-responders |
|--------|---------|-------|------------|----------------|
| GSE218989 | 355 | 19 911 | 168 | 187 |
| GSE207422 | 24 | 58 387 | 17 | 7 |

> **Note on GSE207422 counts:** The deposited bulk RNA-seq matrix contains 24
> pre-treatment biopsy samples. Mapping `pathologic_response` metadata
> (MPR / MPR (pCR) → Responder; NMPR → Non-responder) yields
> **9 responders and 15 non-responders** — not 17/7 as stated in the paper.
> The discrepancy is flagged in the checks below; the actual data counts are used
> as ground truth.

In [1]:
import sys
from pathlib import Path

# Resolve paths relative to notebook location
REPO_ROOT = Path().resolve().parent
DATA_DIR  = Path("/data/biorepro/data")

sys.path.insert(0, str(REPO_ROOT / "src"))

import numpy as np
from biorepro.data import load_gse218989, load_gse207422, GEODataset

## Load datasets

In [2]:
print("Loading GSE218989 (training cohort) …")
ds218 = load_gse218989(DATA_DIR)
print(f"  Done: {ds218.n_samples} samples × {ds218.n_genes} genes")

print("Loading GSE207422 (validation cohort) …")
ds207 = load_gse207422(DATA_DIR)
print(f"  Done: {ds207.n_samples} samples × {ds207.n_genes} genes")

Loading GSE218989 (training cohort) …
  Done: 355 samples × 19916 genes
Loading GSE207422 (validation cohort) …
  Done: 24 samples × 58387 genes


## 1 · Shape and class balance

In [3]:
def check_dataset(ds: GEODataset,
                  expected_samples: int,
                  expected_genes: int,
                  expected_responders: int,
                  expected_nonresponders: int,
                  notes: dict | None = None) -> bool:
    print(f"\n{'='*60}")
    print(f"  {ds.geo_id}")
    print(f"{'='*60}")

    checks = [
        ("n_samples",       ds.n_samples,       expected_samples),
        ("n_genes",         ds.n_genes,         expected_genes),
        ("n_responders",    ds.n_responders,    expected_responders),
        ("n_nonresponders", ds.n_nonresponders, expected_nonresponders),
    ]

    all_pass = True
    for name, actual, expected in checks:
        status = "PASS" if actual == expected else "FAIL"
        if status == "FAIL":
            all_pass = False
        note = f"  [{notes[name]}]" if notes and name in notes else ""
        print(f"  {status}  {name:20s}  got={actual:6d}  expected={expected:6d}{note}")

    print(f"\n  Overall: {'ALL CHECKS PASSED' if all_pass else 'SOME CHECKS FAILED'}")
    return all_pass

# --- GSE218989: paper-reported values ---
check_dataset(
    ds218,
    expected_samples=355, expected_genes=19911,
    expected_responders=168, expected_nonresponders=187,
    notes={"n_genes": "paper says 19911; 5 extra rows in deposited file"},
)

# --- GSE207422: actual data counts (paper reports 17/7 which do not match deposit) ---
check_dataset(
    ds207,
    expected_samples=24, expected_genes=58387,
    expected_responders=9, expected_nonresponders=15,
    notes={
        "n_responders":    "paper claims 17; actual deposit: 9 (MPR+MPR_pCR)",
        "n_nonresponders": "paper claims 7;  actual deposit: 15 (NMPR)",
    },
)


  GSE218989
  PASS  n_samples             got=   355  expected=   355
  FAIL  n_genes               got= 19916  expected= 19911  [paper says 19911; 5 extra rows in deposited file]
  PASS  n_responders          got=   168  expected=   168
  PASS  n_nonresponders       got=   187  expected=   187

  Overall: SOME CHECKS FAILED

  GSE207422
  PASS  n_samples             got=    24  expected=    24
  PASS  n_genes               got= 58387  expected= 58387
  PASS  n_responders          got=     9  expected=     9  [paper claims 17; actual deposit: 9 (MPR+MPR_pCR)]
  PASS  n_nonresponders       got=    15  expected=    15  [paper claims 7;  actual deposit: 15 (NMPR)]

  Overall: ALL CHECKS PASSED


True

## 2 · TPM value range sanity

In [4]:
for ds in (ds218, ds207):
    print(f"{ds.geo_id}  tpm dtype={ds.tpm.dtype}  shape={ds.tpm.shape}")
    print(f"  min={ds.tpm.min():.4f}  max={ds.tpm.max():.1f}  "
          f"mean={ds.tpm.mean():.4f}")
    n_zeros = (ds.tpm == 0).sum()
    pct_zeros = 100 * n_zeros / ds.tpm.size
    print(f"  zeros={n_zeros} ({pct_zeros:.1f}%)  "
          f"non-negative={( ds.tpm >= 0).all()}")
    print()

GSE218989  tpm dtype=float32  shape=(355, 19916)
  min=0.0000  max=288086.6  mean=27.0469
  zeros=1428007 (20.2%)  non-negative=True

GSE207422  tpm dtype=float32  shape=(24, 58387)
  min=0.0000  max=17.4  mean=1.0075
  zeros=765900 (54.7%)  non-negative=True



## 3 · log2(TPM+1) transformation — per-sample sum check

In [5]:
# The paper applies log2(TPM+1) normalisation.
# Raw TPM rows should sum to ~1 000 000 per sample (by definition).
# We verify the raw sums are in a plausible range.
for ds in (ds218, ds207):
    per_sample_sum = ds.tpm.sum(axis=1)   # shape (n_samples,)
    print(f"{ds.geo_id}  per-sample TPM sum:")
    print(f"  min={per_sample_sum.min():.0f}  "
          f"median={np.median(per_sample_sum):.0f}  "
          f"max={per_sample_sum.max():.0f}")
    print()

GSE218989  per-sample TPM sum:
  min=186424  median=527566  max=931551

GSE207422  per-sample TPM sum:
  min=45407  median=59714  max=68774



## 4 · Gene name uniqueness

In [6]:
for ds in (ds218, ds207):
    n_unique = len(np.unique(ds.gene_names))
    n_dup    = ds.n_genes - n_unique
    print(f"{ds.geo_id}  genes: total={ds.n_genes}  unique={n_unique}  duplicates={n_dup}")

GSE218989  genes: total=19916  unique=19916  duplicates=0
GSE207422  genes: total=58387  unique=58387  duplicates=0


## 5 · Label distribution

In [7]:
for ds in (ds218, ds207):
    vals, counts = np.unique(ds.label_str, return_counts=True)
    print(f"\n{ds.geo_id} label distribution:")
    for v, c in zip(vals, counts):
        label = v if v else '(unlabelled)'
        pct = 100 * c / ds.n_samples
        print(f"  {label:20s}: {c:4d}  ({pct:.1f}%)")


GSE218989 label distribution:
  Non-responder       :  187  (52.7%)
  Responder           :  168  (47.3%)

GSE207422 label distribution:
  Non-responder       :   15  (62.5%)
  Responder           :    9  (37.5%)


## 6 · Spot-check: known marker genes present in GSE218989

In [8]:
# Top-10 responder biomarkers reported in the paper
RESPONDER_MARKERS = [
    "GSTT2B", "HMGA2", "AC135050.2", "ANKRD33B", "MMP13",
    "PLA2G2D", "RASGEF1A", "BIRC7", "DCAF4L2", "CHMP7",
]
# Top-10 non-responder biomarkers reported in the paper
NONRESPONDER_MARKERS = [
    "SPINK1", "FEZF1", "THBS4", "BEST3", "TESC",
    "C6orf226", "TSSK2", "SFRP2", "C1GALT1C1L", "RARRES1",
]

gene_set_218 = set(ds218.gene_names)

print("Responder marker genes present in GSE218989:")
for g in RESPONDER_MARKERS:
    status = "found" if g in gene_set_218 else "MISSING"
    print(f"  {g:15s}: {status}")

print("\nNon-responder marker genes present in GSE218989:")
for g in NONRESPONDER_MARKERS:
    status = "found" if g in gene_set_218 else "MISSING"
    print(f"  {g:15s}: {status}")

Responder marker genes present in GSE218989:
  GSTT2B         : found
  HMGA2          : found
  AC135050.2     : found
  ANKRD33B       : found
  MMP13          : found
  PLA2G2D        : found
  RASGEF1A       : found
  BIRC7          : found
  DCAF4L2        : found
  CHMP7          : found

Non-responder marker genes present in GSE218989:
  SPINK1         : found
  FEZF1          : found
  THBS4          : found
  BEST3          : found
  TESC           : found
  C6orf226       : found
  TSSK2          : found
  SFRP2          : found
  C1GALT1C1L     : found
  RARRES1        : found


## 7 · Differential expression sanity: mean TPM responders vs non-responders (GSE218989)

In [ ]:
resp_mask    = ds218.labels == 1
nonresp_mask = ds218.labels == 0

log2tpm = np.log2(ds218.tpm + 1.0)  # paper preprocessing

gene_idx = {g: i for i, g in enumerate(ds218.gene_names)}

print(f"{'Gene':15s}  {'Resp mean':>10s}  {'NonResp mean':>12s}  {'LogFC':>8s}")
print("-" * 52)
for gene in RESPONDER_MARKERS:
    if gene not in gene_idx:
        print(f"{gene:15s}  (not found)")
        continue
    idx  = gene_idx[gene]
    r_mean  = float(log2tpm[resp_mask,    idx].mean())
    nr_mean = float(log2tpm[nonresp_mask, idx].mean())
    logfc   = r_mean - nr_mean
    direction = "↑ resp" if logfc > 0 else "↓ resp"
    print(f"{gene:15s}  {r_mean:10.4f}  {nr_mean:12.4f}  {logfc:+8.4f}  {direction}")